# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | (15) |
| **Integrantes** | (Adolfo Miguel Jimenez) |
| **Caso de estudio** | (Wanderbricks) |
| **Fecha de entrega** | domingo 30 de agosto |
| **🎥 Enlace al video** | *(https://drive.google.com/drive/folders/1RViFDNubZL3syT_PRUfgJxLLGYIyctZa?usp=sharing)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

Wanderbricks cuenta con las tablas `users`, `bookings` y `payments`, que contienen información sobre los usuarios, las reservas y los pagos realizados. Actualmente, estos datos se encuentran separados, por lo que se requiere integrarlos para analizar el comportamiento de los pagos asociados a las reservas.

El proyecto busca construir una base de datos analítica que relacione `users`, `bookings` y `payments` para identificar los medios de pago más utilizados y analizar su comportamiento según el tipo de usuario, país de residencia y valor de la reserva.

La base de datos deberá responder preguntas como: 
- ¿cuál es el medio de pago más utilizado?
- ¿qué porcentaje de los pagos corresponde a cada medio?
- ¿cuál medio de pago representa el mayor valor de las transacciones? 
- ¿existen diferencias en los medios de pago utilizados por usuarios individuales y empresas?


---
## 2. Descripción de los datos

*Volumen, variedad, tipos, calidad observada y relaciones entre tablas.
Esta descripción es la base de la decisión de diseño de la sección 3: sin ella,
cualquier justificación queda en el aire.*

In [0]:
# Exploración inicial del caso
display(spark.sql("SHOW TABLES IN samples.wanderbricks"))

In [0]:
# Conteo de filas por tabla
for t in [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]:
    print(f"{t:30s} {spark.table(f'samples.wanderbricks.{t}').count():>12,}")

## 2. Descripción de los datos

El conjunto de datos seleccionado está compuesto por las tablas `users`, `bookings` y `payments`. `users` contiene 124.509 registros de usuarios, `bookings` contiene 72.250 registros de reservas y `payments` contiene 49.638 registros de pagos. Los datos presentan diferentes tipos como `bigint`, `int`, `float`, `decimal`, `string`, `date`, `timestamp` y `boolean`, correspondientes a identificadores, valores numéricos, fechas, estados y características de los usuarios.

La información es principalmente estructurada y las tablas presentan relaciones mediante identificadores. `users` se relaciona con `bookings` mediante `user_id`, mientras que `bookings` se relaciona con `payments` mediante `booking_id`. Esta estructura permite analizar el flujo usuario → reserva → pago y combinar información de las tres tablas para obtener resultados analíticos sobre el comportamiento de las transacciones.

En cuanto a la calidad, `users` presenta valores nulos en `company_name` (49,8 %) y `email` (0,1 %); la ausencia de `company_name` puede ser válida para usuarios que no son empresas. En `bookings`, la columna `canal` presenta 72.247 valores nulos de 72.250 registros, por lo que tiene una disponibilidad de información muy baja y no resulta relevante para el análisis. `payments` no presenta valores nulos en las columnas analizadas. En conjunto, los datos tienen un esquema definido y permiten relacionar usuarios, reservas y pagos para el análisis propuesto.

---
## 3. Decisiones de diseño y justificación

*Comparar al menos tres paradigmas —relacional, NoSQL (documental / clave-valor / columnar)
y lakehouse— **atando cada criterio a los datos descritos arriba**. Las ventajas genéricas
copiadas de un manual no cuentan.*

| Criterio del caso | Relacional | NoSQL | Lakehouse | Decisión |
|---|---|---|---|---|
| Relación entre `users`, `bookings` y `payments` | Muy adecuado para manejar relaciones mediante `user_id` y `booking_id`. | Menos conveniente para datos que requieren relaciones entre tablas. | Permite relacionar las tablas mediante SQL y Spark. | **Lakehouse** |
| Análisis de reservas y pagos | Permite consultas y agregaciones SQL sobre datos estructurados. | Puede realizar consultas, pero no es la opción más adecuada para este análisis. | Permite análisis mediante SQL y PySpark. | **Lakehouse** |
| Crecimiento de los datos | Puede manejar el volumen actual, pero requiere mayor infraestructura al aumentar el volumen. | Puede escalar horizontalmente. | Permite procesamiento distribuido y escalabilidad. | **Lakehouse** |
| Historial y cambios de los datos | Puede implementarse mediante mecanismos adicionales. | Depende de la tecnología utilizada. | Delta Lake permite manejar versiones, historial y evolución del esquema. | **Lakehouse** |

**Referencias (APA 7):**

---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

In [0]:
CATALOGO = "bigdata_grupo15"   # TODO: reemplazar NN
ESQUEMA  = "wanderbricks"
VOLUMEN  = "datos_crudos"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")
print(f"Trabajando en {CATALOGO}.{ESQUEMA}")

### 4.2 Capa bronce — ingesta de datos crudos

**Creamos la capa de bronce de `payments` `users` y `bookings`**
No creamos un esquema adicional para cada capa, solo en el mismo esquema de wanderbricks creamos cada tabla como se propuso en la clase.
Tambien como está en el notebook se utiliza ingesta_ts permite saber cuándo se realizó la ingesta del registro y origen permite identificar de qué tabla provienen los datos.

In [0]:
from pyspark.sql import functions as F

TABLA_BRONCE = f"{CATALOGO}.{ESQUEMA}.payments_bronce"

bronce = (spark.table("samples.wanderbricks.payments")
            .withColumn("_ingesta_ts", F.current_timestamp())
            .withColumn("_origen", F.lit("samples.wanderbricks.payments")))

(bronce.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(TABLA_BRONCE))

print(f"✅ Tabla creada: {TABLA_BRONCE}")
print(f"   Registros:    {spark.table(TABLA_BRONCE).count():,}")
print(f"   Columnas:     {len(spark.table(TABLA_BRONCE).columns)}")

In [0]:
from pyspark.sql import functions as F

TABLA_BRONCE = f"{CATALOGO}.{ESQUEMA}.bookings_bronce"

bronce = (spark.table("samples.wanderbricks.bookings")
            .withColumn("_ingesta_ts", F.current_timestamp())
            .withColumn("_origen", F.lit("samples.wanderbricks.bookings")))

(bronce.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(TABLA_BRONCE))

print(f"✅ Tabla creada: {TABLA_BRONCE}")
print(f"   Registros:    {spark.table(TABLA_BRONCE).count():,}")
print(f"   Columnas:     {len(spark.table(TABLA_BRONCE).columns)}")

In [0]:
from pyspark.sql import functions as F

TABLA_BRONCE = f"{CATALOGO}.{ESQUEMA}.users_bronce"

bronce = (spark.table("samples.wanderbricks.users")
            .withColumn("_ingesta_ts", F.current_timestamp())
            .withColumn("_origen", F.lit("samples.wanderbricks.users")))

(bronce.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(TABLA_BRONCE))

print(f"✅ Tabla creada: {TABLA_BRONCE}")
print(f"   Registros:    {spark.table(TABLA_BRONCE).count():,}")
print(f"   Columnas:     {len(spark.table(TABLA_BRONCE).columns)}")

### 4.3 Capa plata — datos limpios y tipados

 Justificación de la limpieza `bookings`

En `bookings_bronce` se eliminaron duplicados exactos para evitar que una misma reserva sea contabilizada más de una vez. También se aplicó la regla de que `total_amount` debe ser mayor que cero, debido a que el análisis requiere considerar reservas con un valor válido para relacionarlas posteriormente con los pagos.


In [0]:
from pyspark.sql import functions as F

BRONCE = f"{CATALOGO}.{ESQUEMA}.bookings_bronce"
PLATA = f"{CATALOGO}.{ESQUEMA}.bookings_plata"

# --- Conteo ANTES
n_antes = spark.table(BRONCE).count()

# --- Limpieza
plata = (spark.table(BRONCE)
           # 1. Quitar duplicados exactos
           .dropDuplicates()
           # 2. Regla de negocio: el monto debe ser positivo
           .filter(F.col("total_amount") > 0)
           # 3. Metadato de la capa
           .withColumn("_plata_ts", F.current_timestamp())
        )

n_despues = plata.count()

print(f"Registros en bronce:  {n_antes:,}")
print(f"Registros en plata:   {n_despues:,}")
print(f"Descartados:          {n_antes - n_despues:,}  ({(n_antes-n_despues)/n_antes*100:.2f}%)")

In [0]:
(plata.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(PLATA))

print(f"✅ Tabla creada: {PLATA}")
print(f"   Registros:    {spark.table(PLATA).count():,}")

display(spark.table(PLATA).limit(5))

Justificación de la limpieza `users`

En users_bronce se eliminaron duplicados exactos para evitar que un mismo usuario aparezca repetido y afecte los análisis posteriores. También se validó que user_id no fuera nulo, ya que este identificador permite relacionar users con bookings. No se eliminaron los registros con company_name nulo porque esta ausencia puede ser válida para usuarios que no son empresas.

In [0]:
from pyspark.sql import functions as F

BRONCE = f"{CATALOGO}.{ESQUEMA}.users_bronce"
PLATA = f"{CATALOGO}.{ESQUEMA}.users_plata"

# --- Conteo ANTES
n_antes = spark.table(BRONCE).count()

# --- Limpieza
plata = (spark.table(BRONCE)
           # 1. Quitar duplicados exactos
           .dropDuplicates()
           # 2. Regla de calidad: el identificador del usuario no puede ser nulo
           .filter(F.col("user_id").isNotNull())
           # 3. Metadato de la capa
           .withColumn("_plata_ts", F.current_timestamp())
        )

n_despues = plata.count()

print(f"Registros en bronce:  {n_antes:,}")
print(f"Registros en plata:   {n_despues:,}")
print(f"Descartados:          {n_antes - n_despues:,}  ({(n_antes-n_despues)/n_antes*100:.2f}%)")

In [0]:
(plata.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(PLATA))

print(f"✅ Tabla creada: {PLATA}")
print(f"   Registros:    {spark.table(PLATA).count():,}")

display(spark.table(PLATA).limit(5))

Justificación de la limpieza `payments`

En `payments_bronce` se eliminaron duplicados exactos y se validó que `payment_id` y `booking_id` no fueran nulos, ya que son necesarios para identificar el pago y relacionarlo con una reserva. También se estableció que `amount` debe ser mayor que cero para considerar únicamente pagos con un valor válido para el análisis de los medios de pago.


In [0]:
from pyspark.sql import functions as F

BRONCE = f"{CATALOGO}.{ESQUEMA}.payments_bronce"
PLATA = f"{CATALOGO}.{ESQUEMA}.payments_plata"

# --- Conteo ANTES
n_antes = spark.table(BRONCE).count()

# --- Limpieza
plata = (spark.table(BRONCE)
           # 1. Quitar duplicados exactos
           .dropDuplicates()
           # 2. El identificador del pago no puede ser nulo
           .filter(F.col("payment_id").isNotNull())
           # 3. La reserva asociada no puede ser nula
           .filter(F.col("booking_id").isNotNull())
           # 4. El valor del pago debe ser positivo
           .filter(F.col("amount") > 0)
           # 5. Metadato de la capa
           .withColumn("_plata_ts", F.current_timestamp())
        )

n_despues = plata.count()

print(f"Registros en bronce:  {n_antes:,}")
print(f"Registros en plata:   {n_despues:,}")
print(f"Descartados:          {n_antes - n_despues:,}  ({(n_antes-n_despues)/n_antes*100:.2f}%)")

In [0]:
(plata.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(PLATA))

print(f"✅ Tabla creada: {PLATA}")
print(f"   Registros:    {spark.table(PLATA).count():,}")

display(spark.table(PLATA).limit(5))

### 4.4 Datos semiestructurados

*Al menos una tabla debe manejar estructuras anidadas (structs o arrays).
El clickstream y las reseñas son los candidatos naturales.*

In [0]:
%sql
DESCRIBE samples.wanderbricks.clickstream;

Utilizamos la tabla `clickstream` Se seleccionó clickstream porque contiene datos semiestructurados en la columna metadata, cuyo tipo es struct <device:string,referrer:string>. Esta estructura almacena información adicional sobre cada interacción del usuario, como el dispositivo utilizado y el origen de la visita.

Su uso permite demostrar cómo el Lakehouse y Spark pueden trabajar con estructuras anidadas, accediendo a los campos internos de metadata y posteriormente aplanándolos en columnas independientes para facilitar su consulta y análisis.

In [0]:
display(
    spark.table("samples.wanderbricks.clickstream")
         .select("metadata")
         .limit(10)
)

*Aplanamiento: Se extraen los campos device y referrer contenidos dentro del struct metadata, convirtiéndolos en columnas independientes. Esto facilita las consultas y el análisis posterior de los eventos de navegación.*

In [0]:
plano = spark.table("samples.wanderbricks.clickstream").selectExpr(
    "event",
    "metadata.device       AS device",
    "metadata.referrer     AS referrer",
    "property_id",
    "timestamp",
    "user_id"
)

display(plano)

### 4.5 Propiedades del lakehouse

*Hay que evidenciar las tres: atomicidad, time travel y evolución de esquema.*

In [0]:
TABLA_PLATA = f"{CATALOGO}.{ESQUEMA}.bookings_plata"

In [0]:
antes = spark.table(TABLA_PLATA).count()

spark.sql(f"""
    UPDATE {TABLA_PLATA}
    SET total_amount = total_amount * 1.19
    WHERE status = 'confirmed'
""")

spark.sql(f"""
    DELETE FROM {TABLA_PLATA}
    WHERE total_amount IS NULL
""")

despues = spark.table(TABLA_PLATA).count()

print(f"Registros antes:   {antes:,}")
print(f"Registros después: {despues:,}")
print(f"Eliminados:        {antes - despues:,}")

Time Travel: Se consultó la versión inicial de bookings_plata mediante VERSION AS OF 0 y se comparó con la versión actual. Además, se comparó el monto total de las reservas entre ambas versiones para evidenciar el efecto de las modificaciones realizadas sobre la tabla.

In [0]:
##4.2 El historial
display(spark.sql(f"""
    DESCRIBE HISTORY {TABLA_PLATA}
""").select("version", "timestamp", "operation", "operationMetrics"))

##4.2 Viaje en el tiempo
v0 = spark.sql(
    f"SELECT COUNT(*) AS n FROM {TABLA_PLATA} VERSION AS OF 0"
).first()["n"]

actual = spark.table(TABLA_PLATA).count()

print(f"Versión 0 (original): {v0:,} registros")
print(f"Versión actual:       {actual:,} registros")

##Comparemos el monto total entre las dos versiones. La diferencia es el efecto del UPDATE 
display(spark.sql(f"""
    SELECT 'versión 0'     AS version, ROUND(SUM(total_amount), 0) AS monto
    FROM {TABLA_PLATA} VERSION AS OF 0
    UNION ALL
    SELECT 'versión actual' AS version, ROUND(SUM(total_amount), 0) AS monto
    FROM {TABLA_PLATA}
"""))

Evolución de esquema: Se agregó la nueva columna canal a un conjunto reducido de registros y se utilizó mergeSchema para permitir que Delta incorporara automáticamente esta columna al esquema existente de bookings_plata. Esto demuestra que el modelo puede adaptarse a nuevos atributos sin tener que reconstruir la tabla completa.

In [0]:
# Evolución de esquema con mergeSchema

nuevo = (spark.table(TABLA_PLATA).limit(3)
           .withColumn("canal", F.lit("web"))
           .withColumn("_plata_ts", F.current_timestamp()))

(nuevo.write
   .format("delta")
   .mode("append")
   .option("mergeSchema", "true")
   .saveAsTable(TABLA_PLATA))

print("Columnas después de la evolución:")
for c in spark.table(TABLA_PLATA).columns:
    print(f"  · {c}")

### Conclusión de las propiedades del Lakehouse

En este apartado se evidenciaron tres propiedades de Delta Lake utilizando `bookings_plata`. Primero, se realizaron operaciones de actualización y eliminación para demostrar el manejo de cambios sobre los datos. Luego, mediante `DESCRIBE HISTORY` y `VERSION AS OF 0`, se comprobó que es posible consultar versiones anteriores de la tabla. Finalmente, se utilizó `mergeSchema` para agregar la nueva columna `canal` al esquema existente, demostrando que la estructura de la tabla puede evolucionar sin reconstruirla desde cero.


### 4.6 Consultas analíticas

*Mínimo cinco, en SQL y en PySpark, que respondan preguntas reales del negocio.
Consultas triviales sin conexión con el problema no puntúan.*


##-- Consulta 1 — pregunta que responde: 
##¿Cuál es el método de pago más utilizado por los clientes?


In [0]:
%sql
SELECT
    payment_method,
    COUNT(*) AS cantidad_pagos
FROM payments_plata
GROUP BY payment_method
ORDER BY cantidad_pagos DESC;

In [0]:
from pyspark.sql import functions as F

display(
    spark.table("payments_plata")
         .groupBy("payment_method")
         .agg(F.count("*").alias("cantidad_pagos"))
         .orderBy(F.col("cantidad_pagos").desc())
)

Los resultados muestran que credit_card es el medio de pago más utilizado, con 9.263 pagos, seguido muy de cerca por paypal, con 9.259. El medio menos utilizado es bank_transfer, con 9.103 pagos. La diferencia entre el medio más y menos utilizado es pequeña, por lo que no existe una preferencia ampliamente dominante entre los diferentes medios de pago.

## Consulta 2 — pregunta que responde:
## ¿Cuál es el medio de pago que representa el mayor valor total de pagos?

In [0]:
%sql
SELECT
    payment_method,
    ROUND(SUM(amount), 2) AS valor_total
FROM payments_plata
GROUP BY payment_method
ORDER BY valor_total DESC;

In [0]:
from pyspark.sql import functions as F

display(
    spark.table("payments_plata")
         .groupBy("payment_method")
         .agg(
             F.round(F.sum("amount"), 2).alias("valor_total")
         )
         .orderBy(F.col("valor_total").desc())
)

Los resultados muestran que credit_card representa el mayor valor total de pagos, con aproximadamente 5.134.721,22, seguido de apple_pay con 5.123.009,21. google_pay presenta el menor valor acumulado, con aproximadamente 5.091.275,24. Al igual que en la cantidad de pagos, las diferencias entre los medios son relativamente pequeñas

## Consulta 3 — pregunta que responde:
## ¿Cuál es el estado de los pagos según el medio de pago?

In [0]:
%sql
SELECT
    payment_method,
    status,
    COUNT(*) AS cantidad_pagos
FROM payments_plata
GROUP BY payment_method, status
ORDER BY payment_method, cantidad_pagos DESC;

In [0]:
from pyspark.sql import functions as F

display(
    spark.table("payments_plata")
         .groupBy("payment_method", "status")
         .agg(
             F.count("*").alias("cantidad_pagos")
         )
         .orderBy("payment_method", F.col("cantidad_pagos").desc())
)

**Interpretación:**
Los resultados muestran que la mayoría de los pagos fueron completados en todos los medios de pago. `credit_card` presenta la mayor cantidad de pagos completados, con 9.171, mientras que `bank_transfer` registra la menor cantidad, con 9.000. En cuanto a los pagos fallidos, `bank_transfer` presenta la mayor cantidad, con 103 casos, mientras que `google_pay` presenta la menor, con 90. Esto permite identificar que, aunque los medios de pago tienen un comportamiento general similar, existen pequeñas diferencias en la cantidad de pagos fallidos que pueden ser relevantes para evaluar su funcionamiento.


## Consulta 4 — pregunta que responde:
## ¿Cuál es el valor promedio de las reservas según el medio de pago?

In [0]:
%sql
SELECT
    p.payment_method,
    ROUND(AVG(b.total_amount), 2) AS promedio_reserva,
    COUNT(*) AS cantidad_reservas
FROM bookings_plata b
INNER JOIN payments_plata p
    ON b.booking_id = p.booking_id
GROUP BY p.payment_method
ORDER BY promedio_reserva DESC;

In [0]:
from pyspark.sql import functions as F

display(
    spark.table("bookings_plata").alias("b")
    .join(
        spark.table("payments_plata").alias("p"),
        F.col("b.booking_id") == F.col("p.booking_id"),
        "inner"
    )
    .groupBy("p.payment_method")
    .agg(
        F.round(F.avg("b.total_amount"), 2).alias("promedio_reserva"),
        F.count("*").alias("cantidad_reservas")
    )
    .orderBy(F.col("promedio_reserva").desc())
)

Interpretación:
Los resultados muestran que apple_pay presenta el mayor valor promedio de reserva, con 598,64, mientras que paypal presenta el menor, con 590,04. La diferencia entre ambos es de 8,60. Esto indica que el medio de pago utilizado presenta pequeñas diferencias en el valor promedio de las reservas.

## Consulta 5 — pregunta que responde:
## ¿Qué tipo de usuario realiza más pagos según el medio de pago?

In [0]:
%sql
SELECT
    u.user_type,
    p.payment_method,
    COUNT(*) AS cantidad_pagos
FROM users_plata u
INNER JOIN bookings_plata b
    ON u.user_id = b.user_id
INNER JOIN payments_plata p
    ON b.booking_id = p.booking_id
GROUP BY u.user_type, p.payment_method
ORDER BY u.user_type, cantidad_pagos DESC;

In [0]:
from pyspark.sql import functions as F

display(
    spark.table("users_plata").alias("u")
    .join(
        spark.table("bookings_plata").alias("b"),
        F.col("u.user_id") == F.col("b.user_id"),
        "inner"
    )
    .join(
        spark.table("payments_plata").alias("p"),
        F.col("b.booking_id") == F.col("p.booking_id"),
        "inner"
    )
    .groupBy("u.user_type", "p.payment_method")
    .agg(
        F.count("*").alias("cantidad_pagos")
    )
    .orderBy("u.user_type", F.col("cantidad_pagos").desc())
)

Interpretación:
Los resultados muestran diferencias en los medios de pago según el tipo de usuario. Entre los usuarios business, el medio más utilizado es credit_card, con 4.705 pagos, mientras que entre los usuarios individual, el más utilizado es apple_pay, con 4.665 pagos. Esto permite observar que el comportamiento de pago varía según el tipo de usuario, aunque ambos grupos utilizan los diferentes medios de manera relativamente distribuida.

---
## 5. Resultados

*Qué se obtuvo. Las salidas de las celdas deben quedar visibles en el notebook exportado.*

A partir del procesamiento y análisis de las tablas bookings_plata, payments_plata y users_plata, se obtuvieron resultados relacionados con el comportamiento de las reservas y los medios de pago.

En el análisis de payments_plata se identificó que credit_card fue el medio de pago con mayor cantidad de registros, con 9.263 pagos, seguido de paypal con 9.259. La cantidad de pagos se encuentra distribuida de manera relativamente uniforme entre los diferentes medios.

Al analizar el valor total de los pagos, credit_card presentó el mayor valor acumulado, con 5.134.721,22, mientras que google_pay presentó el menor, con 5.091.275,24.

Respecto al estado de los pagos, la mayoría de las transacciones fueron completadas. credit_card registró 9.171 pagos completados, mientras que bank_transfer presentó la mayor cantidad de pagos fallidos, con 103 casos.

Al relacionar bookings_plata con payments_plata, se observó que apple_pay presentó el mayor valor promedio de reserva, con 598,64, mientras que paypal presentó el menor, con 590,04.

Finalmente, al relacionar users_plata, bookings_plata y payments_plata, se encontraron diferencias según el tipo de usuario. Los usuarios business utilizaron principalmente credit_card, con 4.705 pagos, mientras que los usuarios individual utilizaron principalmente apple_pay, con 4.665 pagos.

Las salidas obtenidas en las consultas SQL y PySpark quedan visibles en el notebook como evidencia de los resultados del análisis.

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

El análisis permitió identificar los principales medios de pago y observar diferencias entre usuarios business e individual. Sin embargo, las diferencias encontradas fueron pequeñas y no permitieron obtener patrones suficientemente relevantes para la toma de decisiones.

Si se desarrollara nuevamente el análisis, sería conveniente incorporar otras tablas, como las de reseñas y características de los hospedajes, para relacionar las calificaciones con el tipo de hospedaje, su ubicación, precio o cantidad de reservas. Esto permitiría obtener información más significativa sobre qué características pueden estar relacionadas con la satisfacción y el comportamiento de los usuarios.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Adolfo Miguel Jiménez Muñoz | Desarrollo completo del proyecto, preparación de los datos, construcción de las capas Bronce y Plata, consultas analíticas y documentación. | Explica el proceso completo, las decisiones de diseño, el código utilizado y los resultados obtenidos. |

Uso de asistentes de IA:
Se utilizó ChatGPT como apoyo para comprender conceptos, revisar y ajustar código y mejorar la redacción del documento. También se utilizó Databricks Genie Code para apoyar la generación, revisión, explicación y corrección del código, así como para identificar y solucionar errores durante el desarrollo en Databricks.

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Por qué eligieron este modelo de datos y qué alternativa descartaron?
2. Muestre una consulta que usted escribió y explique qué hace Spark al ejecutarla.
3. ¿Qué tendrían que cambiar en su diseño si el volumen se multiplicara por cien?

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] Las siete secciones están diligenciadas, no quedaron textos de plantilla
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todos los integrantes aparecen en el video con cámara al presentarse
- [ ] El notebook está confirmado en el repositorio, en la carpeta /ea1
- [ ] El HTML con salidas visibles está subido a Canvas